# Chunk Merge Investigation

**Date:** 2026-03-06  
**chunky-files version:** 2.0.0 (local source at `d804898`)  
**nancy-brain version:** `e932605`  

## Background

When `chunky`'s `SlidingWindowChunker` (or `PythonChunker`) breaks a file into
chunks, some chunks end up very short — single import lines, brief comments,
decorators, or blank separators. The nancy-brain KB build pipeline previously
**dropped** any chunk whose stripped length was below a `MIN_CHUNK_CHARS`
threshold (default 60). This silently removed content from the index, making
files unrecoverable once the `raw/` source was deleted.

This notebook investigates a **forward-merge** strategy: instead of dropping
tiny chunks, prepend them to their *successor* chunk (since imports, comments,
and decorators typically describe what follows). Trailing tiny chunks with no
successor are appended to the last substantial chunk.

### Goals

1. Quantify content loss from the old drop approach vs. the merge approach.
2. Inspect what tiny chunks actually contain.
3. Verify that merged chunks produce readable, semantically coherent units.
4. Confirm 100% content preservation across files of all sizes.

### Pinned Repository Commits

All test files are read from `knowledge_base/raw/` at these commits:

| Repository | Commit | URL |
|---|---|---|
| romancal | `f019333` | https://github.com/spacetelescope/romancal |
| romanisim | `25fe17e` | https://github.com/spacetelescope/romanisim |
| dynesty | `a7af411` | https://github.com/joshspeagle/dynesty |
| webbpsf | `95b37d2` | https://github.com/spacetelescope/webbpsf |
| MulensModel | `69190f5` | https://github.com/rpoleski/MulensModel |
| BAGLE_Microlensing | `e77c6a8` | https://github.com/MovingUniverseLab/BAGLE_Microlensing |

In [1]:
from __future__ import annotations

import dataclasses
import subprocess
from pathlib import Path

from chunky.pipeline import ChunkPipeline
from chunky.types import Chunk, ChunkerConfig, Document

# ── Clone test repos at pinned commits ───────────────────────────────
DATA_DIR = Path("test_data")
DATA_DIR.mkdir(exist_ok=True)

REPOS = {
    "romancal":            ("https://github.com/spacetelescope/romancal.git",            "f019333c3de71c9ef5d9e0e780d66f34cfbc57a9"),
    "romanisim":           ("https://github.com/spacetelescope/romanisim.git",           "25fe17e216e22e6db42216b1f8d0f5cd6307ac10"),
    "dynesty":             ("https://github.com/joshspeagle/dynesty.git",                "a7af41180ed5fa4d42f167581ef41373f388be09"),
    "webbpsf":             ("https://github.com/spacetelescope/webbpsf.git",             "95b37d2263eb60e8d6967239e166f6932724d5fc"),
    "MulensModel":         ("https://github.com/rpoleski/MulensModel.git",               "69190f5db4e8f9ab0f45a39508f7747bddb3547c"),
    "BAGLE_Microlensing":  ("https://github.com/MovingUniverseLab/BAGLE_Microlensing.git","e77c6a8de9fe424fa4abe98486aa52bfb06cc829"),
}

for name, (url, commit) in REPOS.items():
    dest = DATA_DIR / name
    if dest.exists():
        print(f"  ✓ {name} (already cloned)")
        continue
    print(f"  Cloning {name} @ {commit[:7]}…")
    subprocess.run(["git", "clone", "--filter=blob:none", url, str(dest)],
                   capture_output=True, check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", commit],
                   capture_output=True, check=True)
    print(f"  ✓ {name}")

# ── Configuration ────────────────────────────────────────────────────
MIN_CHUNK_CHARS = 60
CONFIG = ChunkerConfig(max_chars=1000, lines_per_chunk=40, line_overlap=5)
PIPELINE = ChunkPipeline()

# ── Test files across size brackets ─────────────────────────────────
TEST_FILES: dict[str, Path] = {
    "dynesty/utils (41 lines)":             DATA_DIR / "dynesty/tests/utils.py",
    "romancal/test_velocity (40 lines)":    DATA_DIR / "romancal/romancal/scripts/tests/test_set_velocity_aberration.py",
    "romancal/update_path (150 lines)":     DATA_DIR / "romancal/romancal/associations/lib/update_path.py",
    "romanisim/test_util (150 lines)":      DATA_DIR / "romanisim/romanisim/tests/test_util.py",
    "dynesty/dynesty (835 lines)":          DATA_DIR / "dynesty/py/dynesty/dynesty.py",
    "MulensModel/uniformcaustic (810 lines)": DATA_DIR / "MulensModel/source/MulensModel/uniformcausticsampling.py",
    "webbpsf/optics (2105 lines)":          DATA_DIR / "webbpsf/webbpsf/optics.py",
}

for label, p in TEST_FILES.items():
    assert p.exists(), f"Missing: {p}"
print(f"\n✅ All {len(TEST_FILES)} test files found")

  Cloning romancal @ f019333…
  ✓ romancal
  Cloning romanisim @ 25fe17e…
  ✓ romanisim
  Cloning dynesty @ a7af411…
  ✓ dynesty
  Cloning webbpsf @ 95b37d2…
  ✓ webbpsf
  Cloning MulensModel @ 69190f5…
  ✓ MulensModel
  Cloning BAGLE_Microlensing @ e77c6a8…
  ✓ BAGLE_Microlensing

✅ All 7 test files found


## 1. The merge function

The forward-merge strategy: accumulate consecutive tiny chunks into a `carry`
buffer, then prepend them to the next substantial chunk. If the file ends with
tiny chunks, append them to the last substantial chunk instead.

This is the candidate implementation for inclusion in `chunky`.

In [2]:
def merge_small_chunks(chunks: list[Chunk], min_chars: int) -> list[Chunk]:
    """Merge any chunk shorter than *min_chars* into its successor.

    Short fragments (imports, comments, decorators, docstrings) typically
    describe what comes *after* them, so they are prepended to the next
    substantial chunk.  A trailing tiny chunk with no successor is appended
    to the last substantial chunk instead.
    """
    if not chunks:
        return chunks
    result: list[Chunk] = []
    carry: Chunk | None = None
    for chunk in chunks:
        if len(chunk.text.strip()) < min_chars:
            if carry is None:
                carry = chunk
            else:
                carry = dataclasses.replace(
                    carry, text=carry.text.rstrip() + "\n" + chunk.text
                )
        else:
            if carry is not None:
                merged_text = carry.text.rstrip() + "\n" + chunk.text
                chunk = dataclasses.replace(chunk, text=merged_text)
                carry = None
            result.append(chunk)
    # Trailing tiny chunk(s) — append to last result
    if carry is not None:
        if result:
            prev = result[-1]
            result[-1] = dataclasses.replace(
                prev, text=prev.text.rstrip() + "\n" + carry.text
            )
        else:
            result.append(carry)
    return result

## 2. Helpers: drop (old approach) and coverage measurement

In [3]:
def drop_small_chunks(chunks: list[Chunk], min_chars: int) -> list[Chunk]:
    """Old approach: silently discard any chunk below the threshold."""
    return [c for c in chunks if len(c.text.strip()) >= min_chars]


def coverage_stats(original: str, chunks: list[Chunk]) -> dict:
    """What fraction of non-blank lines from the original are in the chunks?"""
    original_lines = set()
    for i, line in enumerate(original.splitlines()):
        stripped = line.rstrip()
        if stripped:
            original_lines.add((i, stripped))

    recovered_lines = set()
    for chunk in chunks:
        for line in chunk.text.splitlines():
            stripped = line.rstrip()
            if stripped:
                for orig_idx, orig_line in original_lines:
                    if orig_line == stripped:
                        recovered_lines.add((orig_idx, orig_line))

    total = len(original_lines)
    recovered = len(recovered_lines)
    return {
        "total_non_blank": total,
        "recovered": recovered,
        "missing": total - recovered,
        "pct": (recovered / total * 100) if total else 100.0,
    }

## 3. Merge vs. Drop comparison across all test files

For each file: chunk it, apply both strategies, measure coverage.

In [4]:
results = []
for label, fpath in TEST_FILES.items():
    content = fpath.read_text(errors="replace")
    line_count = len(content.splitlines())
    doc = Document(path=fpath, content=content, metadata={"doc_id": fpath.stem})
    raw = PIPELINE.chunk_documents([doc], config=CONFIG)

    merged = merge_small_chunks(raw, MIN_CHUNK_CHARS)
    dropped = drop_small_chunks(raw, MIN_CHUNK_CHARS)

    cov_merge = coverage_stats(content, merged)
    cov_drop = coverage_stats(content, dropped)
    tiny = [c for c in raw if len(c.text.strip()) < MIN_CHUNK_CHARS]

    results.append({
        "file": label,
        "lines": line_count,
        "raw_chunks": len(raw),
        "merged_chunks": len(merged),
        "dropped_chunks": len(dropped),
        "tiny_count": len(tiny),
        "merge_coverage": cov_merge["pct"],
        "drop_coverage": cov_drop["pct"],
        "drop_lost_lines": cov_drop["missing"],
    })

# ── Summary table ────────────────────────────────────────────────────
header = f"{'File':<42} {'Lines':>5} {'Raw':>4} {'Merge':>5} {'Drop':>5} {'Tiny':>4} {'Merge%':>7} {'Drop%':>7} {'Lost':>4}"
print(header)
print("─" * len(header))
for r in results:
    drop_flag = " ⚠️" if r["drop_lost_lines"] > 0 else ""
    print(
        f"{r['file']:<42} {r['lines']:>5} {r['raw_chunks']:>4} "
        f"{r['merged_chunks']:>5} {r['dropped_chunks']:>5} {r['tiny_count']:>4} "
        f"{r['merge_coverage']:>6.1f}% {r['drop_coverage']:>6.1f}% "
        f"{r['drop_lost_lines']:>4}{drop_flag}"
    )

File                                       Lines  Raw Merge  Drop Tiny  Merge%   Drop% Lost
───────────────────────────────────────────────────────────────────────────────────────────
dynesty/utils (41 lines)                      41    9     4     4    5  100.0%   93.9%    2 ⚠️
romancal/test_velocity (40 lines)             40   14     1     1   13  100.0%   68.8%   10 ⚠️
romancal/update_path (150 lines)             150   20     9     9   11  100.0%   95.7%    5 ⚠️
romanisim/test_util (150 lines)              150   29     9     9   20  100.0%   87.3%   16 ⚠️
dynesty/dynesty (835 lines)                  835   76    50    50   26  100.0%   98.5%   11 ⚠️
MulensModel/uniformcaustic (810 lines)       810   45    39    39    6  100.0%   99.4%    4 ⚠️
webbpsf/optics (2105 lines)                 2105  155   125   125   30  100.0%   99.2%   14 ⚠️


## 4. What are the tiny chunks?

Inspect the content of chunks that fall below the threshold.

In [5]:
for label, fpath in list(TEST_FILES.items())[:4]:  # first 4 files
    content = fpath.read_text(errors="replace")
    doc = Document(path=fpath, content=content, metadata={"doc_id": fpath.stem})
    raw = PIPELINE.chunk_documents([doc], config=CONFIG)
    tiny = [c for c in raw if len(c.text.strip()) < MIN_CHUNK_CHARS]

    if not tiny:
        continue
    print(f"\n{'='*60}")
    print(f"{label}  —  {len(tiny)} tiny chunk(s)")
    print(f"{'='*60}")
    for i, tc in enumerate(tiny):
        preview = tc.text.strip().replace('\n', ' ↵ ')
        if len(preview) > 90:
            preview = preview[:87] + "..."
        print(f"  [{i}] ({len(tc.text.strip()):>3} chars)  {preview!r}")


dynesty/utils (41 lines)  —  5 tiny chunk(s)
  [0] ( 18 chars)  'import numpy as np'
  [1] (  9 chars)  'import os'
  [2] (  0 chars)  ''
  [3] (  0 chars)  ''
  [4] (  0 chars)  ''

romancal/test_velocity (40 lines)  —  13 tiny chunk(s)
  [0] ( 34 chars)  '"""Test set_velocity_aberration"""'
  [1] (  0 chars)  ''
  [2] ( 17 chars)  'import subprocess'
  [3] (  0 chars)  ''
  [4] ( 30 chars)  'import roman_datamodels as rdm'
  [5] ( 25 chars)  'from numpy import isclose'
  [6] ( 19 chars)  '# Testing constants'
  [7] ( 37 chars)  'GOOD_VELOCITY = (100.0, 100.0, 100.0)'
  [8] ( 24 chars)  'GOOD_POS = (359.0, -2.0)'
  [9] ( 37 chars)  'GOOD_SCALE_FACTOR = 1.000316017905845'
  [10] ( 34 chars)  'GOOD_APPARENT_RA = 359.01945099823'
  [11] ( 38 chars)  'GOOD_APPARENT_DEC = -1.980247580394956'
  [12] (  0 chars)  ''

romancal/update_path (150 lines)  —  11 tiny chunk(s)
  [0] ( 46 chars)  '"""Update path of members in an association"""'
  [1] (  0 chars)  ''
  [2] ( 34 chars)  'from os.path

**Observation:** Tiny chunks are almost exclusively:
- Single `import` statements
- Module / function docstrings  
- Short comments (`# Testing constants`)
- Blank separators between sections
- `__all__` declarations

All of these *introduce* what follows — they belong with their successor, not
their predecessor.

## 5. Full chunk dump — verify merge quality

Print every merged chunk for two files (one small, one medium) so we can
manually verify that the groupings are sensible.

In [6]:
INSPECT_FILES = [
    "romancal/test_velocity (40 lines)",
    "romancal/update_path (150 lines)",
]

for label in INSPECT_FILES:
    fpath = TEST_FILES[label]
    content = fpath.read_text(errors="replace")
    doc = Document(path=fpath, content=content, metadata={"doc_id": fpath.stem})
    raw = PIPELINE.chunk_documents([doc], config=CONFIG)
    merged = merge_small_chunks(raw, MIN_CHUNK_CHARS)

    print(f"\n{'#'*70}")
    print(f"# {label}")
    print(f"# Raw: {len(raw)} chunks  →  Merged: {len(merged)} chunks")
    print(f"{'#'*70}")

    for i, chunk in enumerate(merged):
        lines = chunk.text.rstrip().splitlines()
        print(f"\n{'─'*70}")
        print(f"  CHUNK {i+1}/{len(merged)}  ({len(chunk.text.strip())} chars, {len(lines)} lines)")
        print(f"{'─'*70}")
        for line in lines:
            print(f"  {line}")
    print()


######################################################################
# romancal/test_velocity (40 lines)
# Raw: 14 chunks  →  Merged: 1 chunks
######################################################################

──────────────────────────────────────────────────────────────────────
  CHUNK 1/1  (1287 chars, 36 lines)
──────────────────────────────────────────────────────────────────────
  """Test set_velocity_aberration"""
  import subprocess
  import roman_datamodels as rdm
  from numpy import isclose
  
  # Testing constants
  GOOD_VELOCITY = (100.0, 100.0, 100.0)
  GOOD_POS = (359.0, -2.0)
  GOOD_SCALE_FACTOR = 1.000316017905845
  GOOD_APPARENT_RA = 359.01945099823
  GOOD_APPARENT_DEC = -1.980247580394956
  def test_velocity_aberration_script(tmp_path):
      """Test the whole script on a Roman L1 file"""
      path = tmp_path / "velocity_aberration_tmpfile.asdf"
  
      model = rdm.datamodels.ScienceRawModel.create_fake_data(
          {
              "meta": {
             

## 6. Merged chunk size distribution

Verify that merged chunks stay within a healthy range for embedding models
(roughly 100–1200 chars).

In [7]:
all_sizes = []
for label, fpath in TEST_FILES.items():
    content = fpath.read_text(errors="replace")
    doc = Document(path=fpath, content=content, metadata={"doc_id": fpath.stem})
    raw = PIPELINE.chunk_documents([doc], config=CONFIG)
    merged = merge_small_chunks(raw, MIN_CHUNK_CHARS)
    sizes = [len(c.text.strip()) for c in merged]
    all_sizes.extend(sizes)
    print(f"{label:<42}  min={min(sizes):>4}  max={max(sizes):>4}  median={sorted(sizes)[len(sizes)//2]:>4}")

print(f"\n{'─'*60}")
print(f"Overall: {len(all_sizes)} chunks")
print(f"  min={min(all_sizes)}  max={max(all_sizes)}  median={sorted(all_sizes)[len(all_sizes)//2]}")
print(f"  < 100 chars: {sum(1 for s in all_sizes if s < 100)}")
print(f"  100-500:     {sum(1 for s in all_sizes if 100 <= s < 500)}")
print(f"  500-1000:    {sum(1 for s in all_sizes if 500 <= s < 1000)}")
print(f"  > 1000:      {sum(1 for s in all_sizes if s >= 1000)}")

dynesty/utils (41 lines)                    min= 143  max= 371  median= 275
romancal/test_velocity (40 lines)           min=1287  max=1287  median=1287
romancal/update_path (150 lines)            min= 224  max= 737  median= 444
romanisim/test_util (150 lines)             min= 154  max= 878  median= 600
dynesty/dynesty (835 lines)                 min=  67  max=1025  median= 684
MulensModel/uniformcaustic (810 lines)      min= 219  max= 996  median= 878
webbpsf/optics (2105 lines)                 min=  86  max= 995  median= 816

────────────────────────────────────────────────────────────
Overall: 237 chunks
  min=67  max=1287  median=779
  < 100 chars: 6
  100-500:     50
  500-1000:    179
  > 1000:      2


## 7. Conclusion

| Approach | Content preserved | Tiny chunk handling |
|---|---|---|
| **Drop** (old) | Lossy — 1–31% of non-blank lines silently removed | Permanently discarded |
| **Merge forward** (new) | **100%** across all tested files | Prepended to successor chunk |

The merge approach:
- **Preserves all content** — no information is lost from the index.
- **Groups fragments with their context** — imports land with the code that
  uses them; docstrings land with the function they describe.
- **Produces healthy chunk sizes** — mostly 200–1000 chars, well within
  embedding model sweet spots.
- **Reduces total chunk count** — fewer, more meaningful chunks.

### Recommendation

Move `merge_small_chunks` into `chunky` as a public post-processing utility
(e.g. `chunky.pipeline.merge_small_chunks` or a `min_chunk_chars` parameter
on `ChunkerConfig`), then call it from `ChunkPipeline.chunk_documents()`
so every consumer benefits automatically.